# Single-emitter observation-series r-GCN classification

This notebook trains a leakage-safe relational graph classifier on `generated/demo_esm_observation_series.json` for the four diagnostic targets used in `observation_etl_rgcn_end_to_end.ipynb`:

- `aircraft_variant`
- `radar_mode`
- `radar_type`
- `operator_country`

It follows the series concepts in `docs/single_emitter_observation_series.md`: each time-ordered observation remains an individual node, observations are linked by temporal continuity, mode-segment structure is represented without exposing labels as model inputs, and labels are used only as supervised targets/evaluation values.


In [ ]:
from __future__ import annotations

import json, math, random, sys, time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from esm_observation_series_generator import load_observation_series_json
DATA_PATH = ROOT / "generated" / "demo_esm_observation_series.json"
ARTIFACT_DIR = ROOT / "artifacts" / "observation_series_rgcn_classification"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(DATA_PATH)
print(ARTIFACT_DIR)
print(DEVICE)


## Load series and create a label-free inference view

Ground-truth fields are retained in a separate target table, but are recursively stripped from the feature payload before graph construction. This prevents leakage from `ground_truth_label`, `ground_truth_track_label`, and `ground_truth_mode_sequence` into node features or edges.


In [ ]:
LEAKAGE_KEYS = {"ground_truth_label", "ground_truth_track_label", "ground_truth_mode_sequence"}
TARGETS = ["aircraft_variant", "radar_mode", "radar_type", "operator_country"]

def strip_ground_truth(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {k: strip_ground_truth(v) for k, v in obj.items() if k not in LEAKAGE_KEYS}
    if isinstance(obj, list):
        return [strip_ground_truth(v) for v in obj]
    return obj

raw = load_observation_series_json(DATA_PATH)
series_records = raw["observation_series"]
inference_records = strip_ground_truth(series_records)

# Targets are extracted only from the original payload and never merged back into features.
target_rows = []
for s in series_records:
    for obs in s["observations"]:
        gt = obs["ground_truth_label"]
        target_rows.append({
            "observation_id": obs["observation_id"],
            "series_id": obs["series_id"],
            "sequence_index": obs["sequence_index"],
            "aircraft_variant": gt.get("aircraft_variant"),
            "radar_mode": gt.get("mode") or gt.get("radar_mode"),
            "radar_type": gt.get("radar") or gt.get("radar_type"),
            "operator_country": gt.get("operator") or gt.get("operator_country"),
        })

serialized_features = json.dumps(inference_records)
assert "ground_truth" not in serialized_features, "Ground-truth fields leaked into inference view"
print(f"series={len(series_records):,}, observations={len(target_rows):,}")
print(target_rows[0])


## Feature engineering from observations, candidates, temporal context, and segments

Features use measured ESM parameters, approximate kinematics, elapsed time, sensor metadata, and candidate-list aggregate statistics. Candidate values are treated as ambiguous retrieval/candidate evidence, not labels. The segment index below is derived by comparing adjacent candidate radar names and radar-parameter changes in the label-free inference view, so it can mark possible mode transitions without reading true modes.


In [ ]:
def flatten_numeric(prefix: str, value: Any, out: dict[str, float]) -> None:
    if isinstance(value, dict):
        for k, v in value.items(): flatten_numeric(f"{prefix}.{k}" if prefix else k, v, out)
    elif isinstance(value, (int, float)) and not isinstance(value, bool):
        out[prefix] = float(value)

def top_candidate_key(obs: dict[str, Any], key: str) -> str | None:
    cands = obs.get("candidate_labels_from_shared_kg_features") or []
    return cands[0].get(key) if cands else None

def segment_indices(observations: list[dict[str, Any]]) -> list[int]:
    segs, current = [], 0
    prev_radar, prev_freq = None, None
    for obs in observations:
        radar = top_candidate_key(obs, "radar")
        freq = obs.get("esm_radar_parameters", {}).get("frequency_mhz")
        shift = False
        if prev_radar is not None and radar is not None and radar != prev_radar:
            shift = True
        if prev_freq is not None and freq is not None and abs(freq - prev_freq) > 750:
            shift = True
        if segs and shift:
            current += 1
        segs.append(current)
        prev_radar, prev_freq = radar, freq
    return segs

feature_rows, node_meta = [], []
observation_node_indices: list[int] = []
candidates_by_observation_node: dict[int, list[int]] = defaultdict(list)
pending_candidate_nodes: list[tuple[int, dict[str, Any], int, int]] = []
for series in inference_records:
    obs_list = sorted(series["observations"], key=lambda o: o["sequence_index"])
    segs = segment_indices(obs_list)
    n = max(len(obs_list), 1)
    for obs, seg in zip(obs_list, segs):
        obs_node_idx = len(feature_rows)
        observation_node_indices.append(obs_node_idx)
        feats = {
            "node_kind_observation": 1.0,
            "node_kind_candidate": 0.0,
            "elapsed_time_s": float(obs.get("elapsed_time_s", 0.0)),
            "sequence_fraction": float(obs.get("sequence_index", 0)) / max(n - 1, 1),
            "segment_index": float(seg),
            "candidate_count": float(len(obs.get("candidate_labels_from_shared_kg_features") or [])),
            "duration_s": float(series.get("duration_s", 0.0)),
            "observation_count": float(series.get("observation_count", n)),
        }
        flatten_numeric("esm", obs.get("esm_radar_parameters", {}), feats)
        flatten_numeric("kin", obs.get("approximate_kinematics", {}), feats)
        flatten_numeric("loc", obs.get("estimated_emitter_location", {}), feats)
        feature_rows.append(feats)
        node_meta.append({"node_kind": "observation", "observation_id": obs["observation_id"], "series_id": obs["series_id"], "sequence_index": obs["sequence_index"]})
        for rank, candidate in enumerate(obs.get("candidate_labels_from_shared_kg_features") or [], start=1):
            pending_candidate_nodes.append((obs_node_idx, candidate, rank, len(obs.get("candidate_labels_from_shared_kg_features") or [])))

for obs_node_idx, candidate, rank, candidate_count in pending_candidate_nodes:
    rank_fraction = float(rank - 1) / max(candidate_count - 1, 1)
    feats = {
        "node_kind_observation": 0.0,
        "node_kind_candidate": 1.0,
        "candidate_rank": float(rank),
        "candidate_rank_fraction": rank_fraction,
        "candidate_count": float(candidate_count),
    }
    candidate_node_idx = len(feature_rows)
    feature_rows.append(feats)
    candidate_id = f"candidate:{node_meta[obs_node_idx]['observation_id']}:{rank}"
    node_meta.append({
        "node_kind": "candidate",
        "candidate_id": candidate_id,
        "observation_node_idx": obs_node_idx,
        "observation_id": node_meta[obs_node_idx]["observation_id"],
        "series_id": node_meta[obs_node_idx]["series_id"],
        "sequence_index": node_meta[obs_node_idx]["sequence_index"],
        "rank": rank,
        "radar": candidate.get("radar"),
        "aircraft_id": candidate.get("aircraft_id"),
        "operator": candidate.get("operator"),
    })
    candidates_by_observation_node[obs_node_idx].append(candidate_node_idx)

feature_names = sorted({k for row in feature_rows for k in row})
X_np = np.asarray([[row.get(k, 0.0) for k in feature_names] for row in feature_rows], dtype=np.float32)
mu, sigma = X_np.mean(axis=0), X_np.std(axis=0)
sigma[sigma == 0] = 1.0
X = torch.tensor((X_np - mu) / sigma, dtype=torch.float32, device=DEVICE)
print(X.shape, feature_names[:10], {"observation_nodes": len(observation_node_indices), "candidate_nodes": len(feature_rows) - len(observation_node_indices)})


## Build a relational graph

The graph contains observation nodes plus candidate-evidence nodes from the label-free candidate lists. Observation relations capture temporal adjacency (`NEXT_OBSERVATION` / `PREV_OBSERVATION`), same-emitter continuity (`SAME_EMITTER`), inferred same-segment links (`SAME_MODE_SEGMENT`), inferred mode-shift boundaries (`POSSIBLE_MODE_SHIFT`), and self loops. Candidate relations connect observations to ranked candidates and add directed `CONTRADICTS_CANDIDATE` edges from stronger-ranked candidates to incompatible weaker alternatives. No relation is built from target labels.


In [ ]:
RELATIONS = {"self": 0, "next_observation": 1, "prev_observation": 2, "same_emitter": 3, "same_mode_segment": 4, "possible_mode_shift": 5, "has_candidate": 6, "candidate_for": 7, "contradicts_candidate": 8}
edge_src, edge_dst, edge_type = [], [], []

def add_edge(i, j, rel):
    edge_src.append(i); edge_dst.append(j); edge_type.append(RELATIONS[rel])

def candidate_contradiction_reasons(left: dict[str, Any], right: dict[str, Any]) -> list[str]:
    comparisons = {
        "radar": (left.get("radar"), right.get("radar")),
        "aircraft_id": (left.get("aircraft_id"), right.get("aircraft_id")),
        "operator": (left.get("operator"), right.get("operator")),
    }
    return [field for field, (a, b) in comparisons.items() if a is not None and b is not None and a != b]

for i in range(len(node_meta)): add_edge(i, i, "self")

series_to_indices = defaultdict(list)
for i in observation_node_indices: series_to_indices[node_meta[i]["series_id"]].append(i)

segment_by_node = {i: int(feature_rows[i]["segment_index"]) for i in observation_node_indices}
for indices in series_to_indices.values():
    indices = sorted(indices, key=lambda i: node_meta[i]["sequence_index"])
    for a, b in zip(indices, indices[1:]):
        add_edge(a, b, "next_observation"); add_edge(b, a, "prev_observation")
        if segment_by_node[a] == segment_by_node[b]:
            add_edge(a, b, "same_mode_segment"); add_edge(b, a, "same_mode_segment")
        else:
            add_edge(a, b, "possible_mode_shift"); add_edge(b, a, "possible_mode_shift")
    # Sparse same-emitter reinforcement edges between observations two samples apart.
    for a, b in zip(indices, indices[2:]):
        add_edge(a, b, "same_emitter"); add_edge(b, a, "same_emitter")

contradiction_reasons = Counter()
for obs_idx, candidate_indices in candidates_by_observation_node.items():
    candidate_indices = sorted(candidate_indices, key=lambda idx: node_meta[idx]["rank"])
    for candidate_idx in candidate_indices:
        add_edge(obs_idx, candidate_idx, "has_candidate")
        add_edge(candidate_idx, obs_idx, "candidate_for")
    for left_pos, left_idx in enumerate(candidate_indices):
        for right_idx in candidate_indices[left_pos + 1:]:
            reasons = candidate_contradiction_reasons(node_meta[left_idx], node_meta[right_idx])
            if not reasons:
                continue
            add_edge(left_idx, right_idx, "contradicts_candidate")
            contradiction_reasons.update(reasons)

edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long, device=DEVICE)
edge_types = torch.tensor(edge_type, dtype=torch.long, device=DEVICE)
print({name: int((edge_types == rid).sum().cpu()) for name, rid in RELATIONS.items()})
print({"contradiction_reasons": dict(contradiction_reasons)})


## Encode labels and create the required 0.5 / 0.3 / 0.2 train-test-validation split

Splitting is performed by `series_id`, not by individual observation, so adjacent observations from the same emitter track cannot cross from training into evaluation splits.


In [ ]:
label_vocab, y = {}, {}
for task in TARGETS:
    vals = [row[task] for row in target_rows]
    classes = sorted(set(vals))
    label_vocab[task] = classes
    idx = {v: i for i, v in enumerate(classes)}
    y[task] = torch.tensor([idx[v] for v in vals], dtype=torch.long, device=DEVICE)
    print(task, len(classes), Counter(vals).most_common(3))

series_ids = sorted(series_to_indices)
rng = np.random.default_rng(SEED)
rng.shuffle(series_ids)
n = len(series_ids)
n_train = int(round(0.5 * n))
n_test = int(round(0.3 * n))
split_series = {
    "train": set(series_ids[:n_train]),
    "test": set(series_ids[n_train:n_train+n_test]),
    "val": set(series_ids[n_train+n_test:]),
}
splits = {
    name: torch.tensor([i for i in observation_node_indices if node_meta[i]["series_id"] in ids], dtype=torch.long, device=DEVICE)
    for name, ids in split_series.items()
}
print({k: len(v) for k, v in splits.items()})
print({k: len(v) for k, v in split_series.items()})


## Define a deeper anti-overfitting r-GCN with residual blocks and MLP heads

The classifier mirrors the enhanced training configuration used by the package: a five-layer residual r-GCN stack, basis-decomposed relation weights, learned relation gates, layer normalization, dropout, and task-specific MLP heads. The training cell below adds L1 regularization plus validation-loss early stopping with a minimum improvement threshold.

### GELU activation

GELU (Gaussian Error Linear Unit) is a nonlinear activation function: after a linear layer computes a score $x$, GELU transforms that score before it is passed to the next layer. Unlike ReLU, which sets every negative value to exactly zero, GELU acts like a smooth, input-dependent gate that multiplies $x$ by the probability that a standard normal random variable is less than $x$. It is defined as

$$\operatorname{GELU}(x) = x \Phi(x) = x \cdot \frac{1}{2}\left[1 + \operatorname{erf}\left(\frac{x}{\sqrt{2}}\right)\right],$$

where $\Phi(x)$ is the standard normal cumulative distribution function. In practice, GELU keeps large positive activations mostly unchanged, strongly suppresses large negative activations, and smoothly blends values near zero instead of introducing a hard cutoff. This smooth behavior is why GELU is commonly used in modern neural networks and is the activation used in the input projection, residual r-GCN blocks, and task heads below. In PyTorch this is implemented by `torch.nn.GELU()` for module definitions and `torch.nn.functional.gelu()` for tensor operations.


In [ ]:
HIDDEN_DIM = 128
NUM_RGCN_LAYERS = 5
NUM_BASES = 4
DROPOUT = 0.2
TASK_HEAD_HIDDEN_DIM = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
L1_LAMBDA = 1e-5
PATIENCE = 10
EARLY_STOPPING_MIN_DELTA = 1e-4
BATCH_SIZE = 256

class BasisRelGraphConv(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, num_relations: int, num_bases: int | None = None, relation_gates: bool = True):
        super().__init__()
        self.num_relations = max(num_relations, 1)
        self.num_bases = min(num_bases or self.num_relations, self.num_relations)
        self.basis_weights = nn.Parameter(torch.empty(self.num_bases, in_dim, out_dim))
        self.basis_coefficients = nn.Parameter(torch.empty(self.num_relations, self.num_bases))
        self.relation_gate_logits = nn.Parameter(torch.zeros(self.num_relations)) if relation_gates else None
        self.self_loop = nn.Linear(in_dim, out_dim, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_dim))
        nn.init.xavier_uniform_(self.basis_weights)
        nn.init.xavier_uniform_(self.basis_coefficients)

    @property
    def relation_weights(self):
        return torch.einsum("rb,bio->rio", self.basis_coefficients, self.basis_weights)

    def forward(self, x, edge_index, edge_types):
        out = self.self_loop(x)
        if edge_index.numel() == 0:
            return out + self.bias
        src, dst = edge_index
        weights = self.relation_weights
        gates = torch.sigmoid(self.relation_gate_logits).to(dtype=x.dtype) if self.relation_gate_logits is not None else None
        messages = torch.zeros(x.size(0), weights.size(-1), device=x.device, dtype=x.dtype)
        degree = torch.zeros(x.size(0), device=x.device, dtype=x.dtype)
        for rel in range(self.num_relations):
            mask = edge_types == rel
            if not torch.any(mask):
                continue
            rel_msg = x[src[mask]] @ weights[rel]
            if gates is not None:
                rel_msg = rel_msg * gates[rel]
            messages.index_add_(0, dst[mask], rel_msg)
            degree.index_add_(0, dst[mask], torch.ones(mask.sum(), device=x.device, dtype=x.dtype))
        return out + messages / degree.clamp_min(1).unsqueeze(-1) + self.bias

class ResidualRGCNBlock(nn.Module):
    def __init__(self, hidden_dim: int, num_relations: int, num_bases: int, dropout: float):
        super().__init__()
        self.conv = BasisRelGraphConv(hidden_dim, hidden_dim, num_relations, num_bases=num_bases, relation_gates=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_types):
        updated = self.conv(x, edge_index, edge_types)
        updated = F.gelu(updated)
        updated = self.norm(updated)
        updated = self.dropout(updated)
        return x + updated

class SeriesRGCNClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_relations, class_sizes, num_layers=5, num_bases=4, dropout=0.2, task_head_hidden_dim=128):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.layers = nn.ModuleList([
            ResidualRGCNBlock(hidden_dim, num_relations, num_bases, dropout)
            for _ in range(num_layers)
        ])
        self.heads = nn.ModuleDict({
            task: nn.Sequential(
                nn.Linear(hidden_dim, task_head_hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(task_head_hidden_dim, size),
            )
            for task, size in class_sizes.items()
        })

    def encode(self, x, edge_index, edge_types):
        h = self.input_projection(x)
        for layer in self.layers:
            h = layer(h, edge_index, edge_types)
        return h

    def forward(self, x, edge_index, edge_types):
        h = self.encode(x, edge_index, edge_types)
        return {task: head(h) for task, head in self.heads.items()}

model_config = {
    "hidden_dim": HIDDEN_DIM,
    "num_rgcn_layers": NUM_RGCN_LAYERS,
    "num_bases": NUM_BASES,
    "dropout": DROPOUT,
    "task_head_hidden_dim": TASK_HEAD_HIDDEN_DIM,
    "l1_lambda": L1_LAMBDA,
    "patience": PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
    "batch_size": BATCH_SIZE,
}
model = SeriesRGCNClassifier(
    X.size(1),
    HIDDEN_DIM,
    len(RELATIONS),
    {t: len(label_vocab[t]) for t in TARGETS},
    num_layers=NUM_RGCN_LAYERS,
    num_bases=NUM_BASES,
    dropout=DROPOUT,
    task_head_hidden_dim=TASK_HEAD_HIDDEN_DIM,
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print(model)
print(model_config)


## Train with mini-batches, checkpoints, per-epoch diagnostics, and TensorBoard logs

Training uses a `DataLoader` over the training observation-node indices so the supervised loss is optimized in configurable mini-batches. Each mini-batch still runs message passing over the full relational graph to preserve temporal, same-emitter, segment, and candidate-evidence context; only the supervised loss indices are batched. The L1 penalty is scaled by the mini-batch fraction so the total regularization strength remains comparable to full-batch training across one epoch.


In [ ]:
try:
    from torch.utils.tensorboard import SummaryWriter
except Exception:
    SummaryWriter = None

def classification_loss(logits, indices):
    return sum(F.cross_entropy(logits[task][indices], y[task][indices]) for task in TARGETS)

def l1_penalty(model: nn.Module) -> torch.Tensor:
    """Return the L1 norm across all trainable model parameters."""
    return sum(parameter.abs().sum() for parameter in model.parameters())

def accuracy(logits, labels):
    return float((logits.argmax(dim=-1) == labels).float().mean().detach().cpu())

@torch.no_grad()
def split_metrics(split):
    model.eval(); idx = splits[split]; logits = model(X, edge_index, edge_types)
    metrics = {"loss": float(classification_loss(logits, idx).detach().cpu())}
    for task in TARGETS:
        metrics[f"{task}_acc"] = accuracy(logits[task][idx], y[task][idx])
    return metrics

def make_train_loader():
    generator = torch.Generator()
    generator.manual_seed(SEED)
    train_indices = splits["train"].detach().cpu()
    return DataLoader(train_indices, batch_size=BATCH_SIZE, shuffle=True, generator=generator)

EPOCHS = 200
ckpt_path = ARTIFACT_DIR / "best_model.pt"
writer = SummaryWriter(str(ARTIFACT_DIR / "tensorboard")) if SummaryWriter else None
history, best_val, bad_epochs = [], math.inf, 0
train_loader = make_train_loader()
train_size = max(int(splits["train"].numel()), 1)
print(f"training observations={train_size:,}, batch_size={BATCH_SIZE:,}, batches_per_epoch={len(train_loader):,}")

try:
    for epoch in range(1, EPOCHS + 1):
        model.train()
        weighted_train_data_loss = 0.0
        weighted_l1_penalty = 0.0
        for batch_indices in train_loader:
            batch_indices = batch_indices.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(X, edge_index, edge_types)
            data_loss = classification_loss(logits, batch_indices)
            batch_fraction = batch_indices.numel() / train_size
            regularization = L1_LAMBDA * l1_penalty(model) * batch_fraction
            loss = data_loss + regularization
            loss.backward(); optimizer.step()
            weight = batch_indices.numel() / train_size
            weighted_train_data_loss += float(data_loss.detach().cpu()) * weight
            weighted_l1_penalty += float(regularization.detach().cpu())
        row = {"epoch": epoch, "train_data_loss": weighted_train_data_loss, "l1_penalty": weighted_l1_penalty}
        for split in ("train", "test", "val"):
            m = split_metrics(split)
            row.update({f"{split}_{k}": v for k, v in m.items()})
            if writer:
                writer.add_scalar(f"loss/{split}", m["loss"], epoch)
                for task in TARGETS: writer.add_scalar(f"accuracy/{split}_{task}", m[f"{task}_acc"], epoch)
        if writer:
            writer.add_scalar("regularization/l1_penalty", row["l1_penalty"], epoch)
            writer.add_scalar("loss/train_total_with_l1", row["train_data_loss"] + row["l1_penalty"], epoch)
        history.append(row)
        if row["val_loss"] < best_val - EARLY_STOPPING_MIN_DELTA:
            best_val = row["val_loss"]
            bad_epochs = 0
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "label_vocab": label_vocab,
                "feature_names": feature_names,
                "model_config": model_config,
                "splits": {k: v.detach().cpu().tolist() for k, v in splits.items()},
                "history": history,
                "best_val_loss": best_val,
            }, ckpt_path)
        else:
            bad_epochs += 1
        diag = [f"epoch={epoch:04d}", f"train_loss={row['train_loss']:.4f}", f"test_loss={row['test_loss']:.4f}", f"val_loss={row['val_loss']:.4f}", f"batch_train_loss={row['train_data_loss']:.4f}", f"l1={row['l1_penalty']:.4f}", f"bad_epochs={bad_epochs}/{PATIENCE}"]
        diag += [f"train_{task}_acc={row[f'train_{task}_acc']:.3f}" for task in TARGETS]
        diag += [f"test_{task}_acc={row[f'test_{task}_acc']:.3f}" for task in TARGETS]
        print(" | ".join(diag))
        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break
finally:
    if writer: writer.close()
print(f"best checkpoint: {ckpt_path} (val_loss={best_val:.4f})")


## Final evaluation, reports, predictions, and plots


In [ ]:
checkpoint = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
final_metrics = {split: split_metrics(split) for split in ("train", "test", "val")}

model.eval()
with torch.no_grad():
    final_logits = model(X, edge_index, edge_types)
    probs = {task: F.softmax(final_logits[task], dim=-1).detach().cpu().numpy() for task in TARGETS}

predictions = []
for target_idx, node_idx in enumerate(observation_node_indices):
    meta = node_meta[node_idx]
    rec = dict(meta)
    for task in TARGETS:
        pred_idx = int(probs[task][node_idx].argmax())
        rec[f"true_{task}"] = target_rows[target_idx][task]
        rec[f"pred_{task}"] = label_vocab[task][pred_idx]
        rec[f"pred_{task}_confidence"] = float(probs[task][node_idx][pred_idx])
    predictions.append(rec)

summary = {"final_metrics": final_metrics, "best_epoch": int(checkpoint["epoch"]), "split_sizes": {k: int(v.numel()) for k, v in splits.items()}, "split_series_counts": {k: len(v) for k, v in split_series.items()}, "targets": TARGETS, "relations": RELATIONS, "node_counts": {"observations": len(observation_node_indices), "candidates": len(node_meta) - len(observation_node_indices)}, "model_config": checkpoint.get("model_config", model_config), "best_val_loss": float(checkpoint.get("best_val_loss", final_metrics["val"]["loss"])), "leakage_guard": "ground truth keys stripped before feature and graph construction"}
(ARTIFACT_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "predictions.json").write_text(json.dumps(predictions, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))


In [ ]:
epochs = [r["epoch"] for r in history]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for split in ("train", "test", "val"):
    axes[0].plot(epochs, [r[f"{split}_loss"] for r in history], label=split)
axes[0].set_title("Classification loss by split")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("sum cross-entropy"); axes[0].legend(); axes[0].grid(True, alpha=.3)
for task in TARGETS:
    axes[1].plot(epochs, [r[f"test_{task}_acc"] for r in history], label=f"test {task}")
axes[1].set_title("Test accuracy by task")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=.3)
fig.tight_layout()
plot_path = ARTIFACT_DIR / "training_metrics.png"
fig.savefig(plot_path, dpi=150)
plot_path
